# CRA → Vite Migration Tool

Convert old React apps (CRA) to Vite using LLMs.  
Preview refactored configs, migrate safely, and benchmark build times.

In [ ]:
import os
import shutil
import tempfile
import subprocess
import platform
import time
from pathlib import Path
import json
import gradio as gr

system_info = {
    "os": platform.system(),
    "arch": platform.machine(),
    "cpu_cores": os.cpu_count() or 4
}
print("System info:", system_info)

In [ ]:
from dotenv import load_dotenv

load_dotenv(override=True)

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
hf_token = os.getenv("HG_TOKEN")

for name, key in [("OpenRouter", openrouter_api_key), ("Hugging Face", hf_token)]:
    if key:
        print(f"{name} API Key loaded, starts with: {key[:8]}...")
    else:
        print(f"{name} API Key not set!")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

HF_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL, use_auth_token=hf_token)
hf_model = AutoModelForCausalLM.from_pretrained(HF_MODEL, use_auth_token=hf_token)

hf_pipeline = pipeline(
    "text-generation",
    model=hf_model,
    tokenizer=tokenizer,
    device_map="auto",
    max_new_tokens=1024
)
print(f"Hugging Face model '{HF_MODEL}' loaded successfully.")

In [ ]:
from openai import OpenAI

openrouter_url = "https://openrouter.ai/api/v1"
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

In [ ]:
HF_LLAMA = "huggingface/llama"
OPENROUTER_GPT = "openai/gpt-4o-mini"
DEFAULT_MODEL = OPENROUTER_GPT

clients = {
    HF_LLAMA: hf_pipeline,
    OPENROUTER_GPT: openrouter
}

In [ ]:
system_prompt = """
Your task is to convert a React app created with CRA into a Vite project.
Keep all behavior identical, update scripts, dependencies, configs, and imports.
Only output changed files and content as valid JSON: {"files": {"path/to/file": "new content"}}
"""

def user_prompt_for(project_path: str) -> str:
    return (
        f"Convert this CRA project at path: {project_path} to Vite. "
        "Include package.json scripts and vite.config.js updates. Output JSON only."
    )

def messages_for(project_path: str) -> list:
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(project_path)}
    ]

In [ ]:
def convert_cra_to_vite(model: str, project_path: str) -> dict:
    client = clients.get(model)
    if not client:
        return {"error": "Model not available"}

    if model == HF_LLAMA:
        prompt = f"{system_prompt}\n\n{user_prompt_for(project_path)}"
        output = client(prompt)[0]["generated_text"]
    else:
        # OpenRouter
        response = client.chat.completions.create(
            model=model,
            messages=messages_for(project_path),
            timeout=180
        )
        output = response.choices[0].message.content.strip()

    try:
        files_json = json.loads(output).get("files", {})
    except Exception:
        return {"error": f"LLM output parsing error: {output}"}

    return files_json

In [ ]:
def apply_migration(files_json: dict, project_path: str) -> str:
    temp_dir = tempfile.mkdtemp()
    shutil.copytree(project_path, temp_dir, dirs_exist_ok=True)

    for rel_path, content in files_json.items():
        full_path = os.path.join(temp_dir, rel_path)
        os.makedirs(os.path.dirname(full_path), exist_ok=True)
        with open(full_path, "w", encoding="utf-8") as f:
            f.write(content)

    return temp_dir

In [ ]:
def benchmark_project(path: str) -> str:
    try:
        start = time.perf_counter()
        subprocess.run(["npm", "install"], cwd=path, check=True, capture_output=True)
        subprocess.run(["npm", "run", "build"], cwd=path, check=True, capture_output=True)
        elapsed = time.perf_counter() - start
        return f"Build completed in {elapsed:.2f}s"
    except subprocess.CalledProcessError as e:
        return f"Error during build: {e.stderr.decode()}"

In [ ]:
CSS = """
:root {
  --accent: #753991;
  --card: #161a22;
  --text: #e9eef5;
}
.gradio-container { max-width: 100% !important; padding: 0 40px !important; }
.convert-btn button { background: var(--accent) !important; color: white !important; font-weight: 700; }
"""

def run_migration(model: str, project_path: str):
    files_json = convert_cra_to_vite(model, project_path)
    if "error" in files_json:
        return files_json["error"], "", ""
    
    temp_dir = apply_migration(files_json, project_path)
    benchmark_msg = benchmark_project(temp_dir)
    return json.dumps(files_json, indent=2), f"Migration temp dir: {temp_dir}", benchmark_msg

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="CRA → Vite Migrator") as ui:
    gr.Markdown("## CRA → Vite Migration Tool")

    project_path_input = gr.Textbox(label="CRA Project Path", placeholder="/path/to/cra/project")
    model_input = gr.Dropdown([HF_LLAMA, OPENROUTER_GPT], value=DEFAULT_MODEL, label="LLM Model")

    migrate_btn = gr.Button("Migrate & Benchmark", elem_classes=["convert-btn"])
    files_output = gr.Code(label="Converted Files (JSON)", language="json", lines=20)
    temp_output = gr.Textbox(label="Temporary Migration Directory")
    benchmark_output = gr.Textbox(label="Build Benchmark")

    migrate_btn.click(
        fn=run_migration,
        inputs=[model_input, project_path_input],
        outputs=[files_output, temp_output, benchmark_output]
    )

ui.launch()